# Retry a tool after a temporary failure

Try an order lookup that fails twice before succeeding. LiteAgents retries it without asking you to rerun the agent.

Run the cells in order. You need an [OpenAI API key](https://platform.openai.com/api-keys) with API credit.

[Open in Colab](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/recipes/05_retries.ipynb)

## 1. Install

Install LiteAgents and the integrations used in this notebook.

In [ ]:
%pip install -q --progress-bar off "liteagents[pydantic-ai] @ https://github.com/BerriAI/liteagents/releases/download/v0.3.0a5/liteagents-0.3.0a5-py3-none-any.whl"

## 2. Add your key

Run this cell, paste your key into the hidden input, and press Enter.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = (os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")).strip()
if not os.environ["OPENAI_API_KEY"]:
    raise ValueError("Run this cell again and enter your OpenAI API key.")

## 3. Make an unreliable tool

This deliberately raises `TimeoutError` on the first two attempts and returns a total on the third.

In [ ]:
from typing import ClassVar

from liteagents import Tool, operation_id


class UnstableLookup(Tool):
    name = "unstable_lookup"
    description = "Return the confirmed total for order A123."
    input_schema: ClassVar[dict] = {"type": "object", "properties": {}}

    def __init__(self):
        self.attempts = 0
        self.keys = []

    async def execute(self, input):
        self.attempts += 1
        self.keys.append(operation_id())
        print("Lookup attempt", self.attempts)
        if self.attempts < 3:
            raise TimeoutError("Demo: the order service is temporarily unavailable")
        return "Confirmed total: USD 12"

## 4. Enable retries and run

Allow up to three attempts. You should see attempts **1, 2, 3**, then an answer with the total.

In [ ]:
from liteagents import ProfileOptions, RecoveryOptions, run

profile = ProfileOptions(
    harness="pydantic-ai",
    model="openai/gpt-5.4-mini",
)
profile.recovery = RecoveryOptions(retries={"max_attempts": 3})
tool = UnstableLookup()
result = await run("Call unstable_lookup once and report the total.", profile=profile, tools=[tool])
print(result.text)

## 5. Check the operation key

All retries of one operation share a key. A tool that writes to an external service can pass it as that service’s idempotency key to avoid duplicate writes.

In [ ]:
print("Attempts:", tool.attempts)
print("Distinct operation keys:", len(set(tool.keys)))

Try setting `max_attempts` to `2`: the same tool will fail before reaching its successful third attempt. Restore `3` to let it complete.

Retries alone do not prevent duplicate external effects; the external service must honor your idempotency key.

[Other model providers](https://github.com/BerriAI/liteagents/blob/main/docs/models.md) · [All cookbooks](https://github.com/BerriAI/liteagents/blob/main/cookbook/README.md)